<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/Ball_Trap_Sparse_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ball Trap Brain — Trace-Guided Sparse Discrete Trainer

This notebook preserves the original `Ball_Trap_Brain` class and adds a trainer that searches its existing discrete parameters directly. It does **not** add layers, change `compute_input`, or replace the discrete architecture with a relaxation.

## Why this trainer is different

- **Exact behavior:** each candidate is evaluated with the original discrete routing/energy rules.
- **Faster exact travel calculation:** all per-token offsets are calculated in O(L), using a weighted-sum identity equivalent to the original O(L²) repeated sum.
- **Sparse updates:** a deflector update is scored only on examples whose current trajectories hit it; a token parameter update is scored only on examples containing that token.
- **Bounded trajectory memory:** full path traces are cached only for the current minibatch. The inverse index of active nodes is sparse, and token IDs are stored in packed integer arrays.
- **Discrete search:** it proposes destination/value pairs, alphabet start positions, and integer directions. It uses sampled candidate screening followed by exact rescoring on the affected minibatch cases.
- **Explicit model selection:** a size sweep picks the smallest tested brain within a validation-loss tolerance of the best tested model.

The method is a heuristic optimizer, not a proof of global optimality. Brain-size behavior is not guaranteed to be monotonic in this architecture, so size search evaluates each supplied size rather than using an unjustified binary search. Optional multiple random restarts reduce sensitivity to initialization.

Run the first cells top-to-bottom. The architecture cells below are copied from `Ball_Trap_Brain(2).ipynb` unchanged.


In [ ]:
import random
import math
import copy

def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        # Loop to find ALL occurrences of the item instead of just the first
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break

            matches.append((index, -len(item_str), order, item))

            # Move start position forward by 1 to continue searching the rest of the string
            # (Adding 1 instead of len(item_str) allows it to safely catch overlapping patterns)
            start_pos = index + 1

    # Sorts by starting index, then by longest string, then by original order
    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0

    for match in matches:
        start_index = match[0]
        length = -match[1]  # The length was stored as a negative number
        item = match[3]

        # Only add the token if it doesn't overlap with the previously added token
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length
    #print(non_overlapping_items)
    return non_overlapping_items

In [ ]:
class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1, direction_range=10):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce*2
        self.alphabet = alphabet
        # Each alphabet token gets its own integer direction in
        # [-direction_range, direction_range] instead of only +-1. A wider range
        # gives travel()'s leave-one-out, position-weighted sum far more
        # resolution to separate tokens by -- with only +-1, two different tokens
        # were indistinguishable to the sum beyond their single bit of sign, which
        # was a capacity bottleneck independent of (and compounding) the
        # order-invariance one that position weighting alone fixes.
        self.direction_range = direction_range
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors-1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes (self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
          for j in range(holes_multiplier):
            self.deflectors_list[random.randint(0, deflectors-1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        return [random.randint(-self.direction_range, self.direction_range) for _ in range(alphabet)]

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors-1) for i in range(alphabet)]


    def compute_input(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))


        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return output

    def compute_deflectors(self, input):
        """
        Processes input exactly like compute_input but returns an array
        of all deflectors hit during the ball's travel.
        """
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))

        # Passes the standard cap of 10 and the current ball_energy
        return self.ricochet_deflectors(travel, 10, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        """
        Modified ricochet logic that records and returns the deflectors
        hit instead of the output values.
        """
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]

                # Append the deflector's details to our tracking array
                deflectors_hit.append(new_idx)

                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return deflectors_hit

    def travel(self, directions, position, index):
        # Linear position weighting: weight token j's direction by (j+1) instead of
        # a flat, order-invariant sum. This means swapping two positions changes the
        # total (unless their directions happen to be equal), breaking the old
        # collision where every anagram of the same characters produced the exact
        # same travel value -- and therefore the exact same output -- regardless of
        # order. Skip index-in-place rather than pop()+re-sum, so the weights (j+1)
        # still refer to true sequence positions rather than shifted ones.

        # Normalized by the sum of weights actually used (i.e. excluding `index`)
        # instead of left as a raw sum. With a raw sum, the weight (j+1) grows with
        # sequence length, so `total`'s typical magnitude grows roughly quadratically
        # with the number of tokens (worst case exactly quadratic, when directions are
        # all the same sign) -- for long inputs this blows `total` up to many multiples
        # of len(deflectors_list), so the `% n` in ricochet() effectively randomizes the
        # starting deflector regardless of actual token content, and the raw, un-wrapped
        # `distance` hop at that near-random start can drain most of ball_energy on the
        # very first step. Both effects make long sequences increasingly likely to never
        # reach a hole at all, collapsing the output down to only the few trajectories
        # that get lucky -- often just one. Dividing by the weight sum keeps `total`'s
        # magnitude bounded to roughly [-direction_range, direction_range] regardless of
        # sequence length (it's a weighted *average*, not a weighted *sum*), while still
        # preserving order-sensitivity: swapping two positions still changes the result
        # whenever their directions differ.
        total = 0
        weight_sum = 0
        for j, d in enumerate(directions):
            if j == index:
                continue
            weight = j + 1
            total += d * weight
            weight_sum += weight
        if weight_sum > 0:
            # Round back to an int: `position` (and the value this function returns)
            # is used directly as a deflector index downstream (`t % n` in ricochet()),
            # which requires an int -- the normalization above only needs to keep the
            # *magnitude* bounded, not switch the representation to floats.
            total = round(total / weight_sum)
        position += total
        return position

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

    def clone(self):
      # Create a lightweight dummy instance
      new_brain = Ball_Trap_Brain(1, self.ball_energy // 2, self.alphabet, self.output,
                                   self.holes_multiplier, self.direction_range)

      # Fast shallow copy of the 10,000 elements
      new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
      new_brain.alphabet_directions = self.alphabet_directions[:]
      new_brain.alphabet_start_positions = self.alphabet_start_positions[:]

      return new_brain

    def shift_alphabet(self):
        random.shuffle(self.alphabet)
        for i in range(len(self.deflectors_list)):
          if self.deflectors_list[i][1] in self.output:
            idx = self.output.index(self.deflectors_list[i][1])
            self.deflectors_list[i][1] = self.alphabet[idx]

    def Mutate(self, individual_probability=0.1, probability=0.05):
        if random.random() < individual_probability:
            for i in range(len(self.deflectors_list)):
                  if random.random() < probability:
                      self.deflectors_list[i][0] = random.randint(0, len(self.deflectors_list)-1)

            for i in range(len(self.deflectors_list)):
                 if random.random() < probability:
                     self.deflectors_list[random.randint(0, len(self.deflectors_list)-1)][1] = random.choice(self.output)

            for i in range(len(self.alphabet_directions)):
                 if random.random() < probability:
                     self.alphabet_directions[i] = random.randint(-self.direction_range, self.direction_range)

            for i in range(len(self.alphabet_start_positions)):
                  if random.random() < probability:
                      self.alphabet_start_positions[i] = random.randint(0, len(self.deflectors_list)-1)

            if random.random() < probability:
                  self.set_holes(self.output, len(self.deflectors_list), self.holes_multiplier)

        return self

## Trainer implementation

The following code defines `SparseTraceTrainer` and `search_brain_sizes`. It can also be downloaded separately as `ball_trap_sparse_trainer.py` and imported from another notebook that already defines the original brain class.


In [ ]:
"""Trace-guided discrete trainer for the original Ball_Trap_Brain architecture.

This module does not add layers or change the simulator.  It only updates the
existing parameters: deflectors_list, alphabet_directions, and
alphabet_start_positions.

Requires a Ball_Trap_Brain instance and (input_text, target_text) pairs.
The exact discrete simulator is reimplemented here for fast scoring; call
verify_simulator() to assert equivalence with brain.compute_input().
"""
from __future__ import annotations

import math
import random
from collections import Counter
from array import array
from itertools import islice
from typing import Any, Callable, Iterable, Optional, Sequence

try:
    from rapidfuzz.distance import Levenshtein as _Levenshtein
except Exception:  # pragma: no cover - fallback for minimal environments
    _Levenshtein = None


def _tokenize_input(input_value: Any, alphabet: Sequence[Any]) -> list[Any]:
    """Match the tokenizer in Ball_Trap_Brain(2).ipynb exactly."""
    input_str = str(input_value)
    matches = []
    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break
            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1
    matches.sort(key=lambda match: (match[0], match[1], match[2]))
    tokens = []
    last_end_index = 0
    for start_index, neg_len, _, item in matches:
        length = -neg_len
        if start_index >= last_end_index:
            tokens.append(item)
            last_end_index = start_index + length
    return tokens


def _fallback_normalized_edit_distance(a: str, b: str) -> float:
    if a == b:
        return 0.0
    if not a:
        return 1.0 if b else 0.0
    if not b:
        return 1.0
    # Low-memory exact Levenshtein distance: O(min(len(a), len(b))) memory.
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1] + 1, prev[j] + 1,
                           prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1] / max(len(a), len(b))


def normalized_edit_distance(prediction: str, target: str) -> float:
    """0 is exact; 1 is maximally distant. RapidFuzz is optional but recommended."""
    prediction, target = str(prediction), str(target)
    if _Levenshtein is not None:
        return float(_Levenshtein.normalized_distance(prediction, target))
    return _fallback_normalized_edit_distance(prediction, target)


def _brain_snapshot(brain: Any) -> dict[str, Any]:
    return {
        "deflectors_list": [[d[0], d[1]] for d in brain.deflectors_list],
        "alphabet_directions": list(brain.alphabet_directions),
        "alphabet_start_positions": list(brain.alphabet_start_positions),
    }


def _restore_snapshot(brain: Any, snapshot: dict[str, Any]) -> None:
    brain.deflectors_list = [[d[0], d[1]] for d in snapshot["deflectors_list"]]
    brain.alphabet_directions = list(snapshot["alphabet_directions"])
    brain.alphabet_start_positions = list(snapshot["alphabet_start_positions"])


class SparseTraceTrainer:
    """Exact discrete, minibatch, trace-guided optimizer for Ball_Trap_Brain.

    Parameters
    ----------
    brain:
        An instance of the existing Ball_Trap_Brain class. It is optimized in
        place; its architecture and inference methods are not modified.
    cases:
        Iterable of (input_text, target_text) pairs.
    batch_size:
        Maximum number of examples whose paths are kept in working memory.
    candidate_budget:
        Maximum number of discrete alternatives scored approximately per move.
    proposal_rows:
        Number of affected examples used to shortlist candidates.
    finalists:
        Number of shortlisted alternatives re-scored on all affected examples in
        the current batch before accepting a move.
    seed:
        Local RNG seed; model initialization is not changed by this seed.
    """

    def __init__(
        self,
        brain: Any,
        cases: Iterable[tuple[Any, Any]],
        *,
        batch_size: int = 64,
        candidate_budget: int = 24,
        proposal_rows: int = 24,
        finalists: int = 2,
        temperature: float = 0.025,
        neutral_move_probability: float = 0.015,
        seed: int = 0,
    ) -> None:
        self.brain = brain
        self.rng = random.Random(seed)
        self.batch_size = max(1, int(batch_size))
        self.candidate_budget = max(4, int(candidate_budget))
        self.proposal_rows = max(1, int(proposal_rows))
        self.finalists = max(1, int(finalists))
        self.initial_temperature = max(0.0, float(temperature))
        self.neutral_move_probability = min(1.0, max(0.0, neutral_move_probability))
        self.alphabet = list(brain.alphabet)
        self.D = len(brain.deflectors_list)
        if self.D < 1:
            raise ValueError("The brain must have at least one deflector.")
        if len(brain.alphabet_directions) != len(self.alphabet):
            raise ValueError("alphabet_directions must match the brain alphabet.")
        if len(brain.alphabet_start_positions) != len(self.alphabet):
            raise ValueError("alphabet_start_positions must match the brain alphabet.")
        self._rebuild_value_indexes()
        self.cases = self._prepare_cases(cases)
        if not self.cases:
            raise ValueError("At least one training case is required.")
        self.history: list[dict[str, Any]] = []
        self.steps_taken = 0
        self.accepted_moves = 0

    def _rebuild_value_indexes(self) -> None:
        self.integer_counts = Counter(
            value for _, value in self.brain.deflectors_list if isinstance(value, int)
        )
        # Output holes are sparse in this architecture (the constructor places only
        # len(output) * holes_multiplier of them), so index non-integer values only.
        self.hole_nodes: dict[tuple[str, str], set[int]] = {}
        for node, (_, value) in enumerate(self.brain.deflectors_list):
            if not isinstance(value, int):
                self.hole_nodes.setdefault(self._value_key(value), set()).add(node)

    def _prepare_cases(self, cases: Iterable[tuple[Any, Any]]) -> list[dict[str, Any]]:
        prepared = []
        for item in cases:
            if len(item) != 2:
                raise ValueError("Each case must be an (input, target) pair.")
            input_value, target_value = item
            tokens = _tokenize_input(input_value, self.alphabet)
            # Match compute_input's alphabet.index(token) lookup behavior.
            token_ids_list = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
            # Packed integer arrays are substantially smaller than tuples of Python
            # ints for corpora with millions of input tokens.
            typecode = "H" if len(self.alphabet) <= 65536 else "I"
            token_ids = array(typecode, token_ids_list)
            prepared.append({
                "input": str(input_value),
                "target": str(target_value),
                "token_ids": token_ids,
                "alpha_set": frozenset(token_ids),
            })
        return prepared

    def _context_offsets(self, token_ids: Sequence[int]) -> list[int]:
        """Compute the exact integer offsets used by brain.travel()."""
        directions = [self.brain.alphabet_directions[a] for a in token_ids]
        # The original travel() recomputes a weighted sum for every token, O(L^2).
        # Since each output ball excludes only its own term, compute the full
        # weighted sum once and subtract that term: exact same integer arithmetic,
        # but O(L) work for all offsets together.
        length = len(directions)
        weighted_total = sum(direction * (j + 1) for j, direction in enumerate(directions))
        total_weight = length * (length + 1) // 2
        offsets = []
        for i, direction in enumerate(directions):
            weight_sum = total_weight - (i + 1)
            total = weighted_total - direction * (i + 1)
            offsets.append(round(total / weight_sum) if weight_sum > 0 else 0)
        return offsets

    def _simulate_prepared(
        self, example: dict[str, Any], *, collect_nodes: bool = False
    ) -> tuple[str, set[int]]:
        """Mirror compute_input + travel + ricochet with the current parameters."""
        token_ids = example["token_ids"]
        if not token_ids:
            return "", set()
        offsets = self._context_offsets(token_ids)
        starts = self.brain.alphabet_start_positions
        deflectors = self.brain.deflectors_list
        n = len(deflectors)
        ball_energy = self.brain.ball_energy
        emitted: list[str] = []
        visited: set[int] = set()
        for i, alpha_id in enumerate(token_ids):
            node = (starts[alpha_id] + offsets[i]) % n
            energy = ball_energy
            cap_idx = 0
            while energy > 0 and cap_idx < 10:
                if collect_nodes:
                    visited.add(node)
                dest, value = deflectors[node]
                distance = abs(dest - node)
                if not isinstance(value, int):
                    emitted.append(value)
                    energy += 1
                    energy -= distance
                else:
                    energy += value
                    energy -= distance
                node = dest
                cap_idx += 1
        return "".join(emitted), visited

    def verify_simulator(self, inputs: Optional[Iterable[Any]] = None) -> bool:
        """Raise AssertionError if this trainer's simulator diverges from compute_input."""
        if inputs is None:
            inputs = [case["input"] for case in self.cases[: min(100, len(self.cases))]]
        for input_value in inputs:
            expected = "".join(self.brain.compute_input(input_value))
            prepared = self._prepare_cases([(input_value, "")])[0]
            actual, _ = self._simulate_prepared(prepared, collect_nodes=False)
            if actual != expected:
                raise AssertionError(
                    f"Simulator mismatch for {input_value!r}: {expected!r} != {actual!r}"
                )
        return True

    def evaluate(self, cases: Optional[Iterable[tuple[Any, Any]]] = None) -> dict[str, Any]:
        prepared = self.cases if cases is None else self._prepare_cases(cases)
        if not prepared:
            return {"n": 0, "loss": float("nan"), "exact_match": float("nan")}
        losses = []
        exact = 0
        for example in prepared:
            prediction, _ = self._simulate_prepared(example)
            target = example["target"]
            losses.append(normalized_edit_distance(prediction, target))
            exact += prediction == target
        return {
            "n": len(prepared),
            "loss": sum(losses) / len(losses),
            "exact_match": exact / len(prepared),
        }

    def _make_batch_cache(self, case_ids: Sequence[int]) -> dict[str, Any]:
        cache = {
            "case_ids": list(case_ids),
            "predictions": [],
            "losses": [],
            "row_nodes": [],
            # Sparse inverted index: no empty set allocated for every deflector.
            "node_rows": {},
            "active_nodes": [],
            "active_pos": {},
        }
        for local_row, case_id in enumerate(cache["case_ids"]):
            pred, nodes = self._simulate_prepared(self.cases[case_id], collect_nodes=True)
            loss = normalized_edit_distance(pred, self.cases[case_id]["target"])
            cache["predictions"].append(pred)
            cache["losses"].append(loss)
            cache["row_nodes"].append(nodes)
            for node in nodes:
                cache["node_rows"].setdefault(node, set()).add(local_row)
        cache["active_nodes"] = list(cache["node_rows"].keys())
        cache["active_pos"] = {node: i for i, node in enumerate(cache["active_nodes"])}
        return cache

    def _refresh_rows(self, cache: dict[str, Any], rows: Sequence[int]) -> None:
        for local_row in sorted(set(rows)):
            for node in cache["row_nodes"][local_row]:
                rows_for_node = cache["node_rows"].get(node)
                if rows_for_node is not None:
                    rows_for_node.discard(local_row)
                    if not rows_for_node:
                        del cache["node_rows"][node]
                        pos = cache["active_pos"].pop(node)
                        last_node = cache["active_nodes"].pop()
                        if pos < len(cache["active_nodes"]):
                            cache["active_nodes"][pos] = last_node
                            cache["active_pos"][last_node] = pos
            case_id = cache["case_ids"][local_row]
            pred, nodes = self._simulate_prepared(self.cases[case_id], collect_nodes=True)
            cache["predictions"][local_row] = pred
            cache["losses"][local_row] = normalized_edit_distance(
                pred, self.cases[case_id]["target"]
            )
            cache["row_nodes"][local_row] = nodes
            for node in nodes:
                rows_for_node = cache["node_rows"].get(node)
                if rows_for_node is None:
                    cache["node_rows"][node] = {local_row}
                    cache["active_pos"][node] = len(cache["active_nodes"])
                    cache["active_nodes"].append(node)
                else:
                    rows_for_node.add(local_row)

    def _sample_rows(self, rows: Sequence[int], cap: int) -> list[int]:
        if len(rows) <= cap:
            return list(rows)
        return self.rng.sample(list(rows), cap)

    @staticmethod
    def _value_key(value: Any) -> tuple[str, str]:
        return type(value).__name__, repr(value)

    def _target_symbols(self, local_rows: Sequence[int], cache: dict[str, Any], limit: int = 4) -> list[Any]:
        if not local_rows:
            return []
        targets = [self.cases[cache["case_ids"][r]]["target"] for r in local_rows]
        counts = []
        for symbol in self.brain.output:
            if isinstance(symbol, int):
                continue  # integer values are interpreted as energy, not emitted symbols
            symbol_str = str(symbol)
            if not symbol_str:
                continue
            frequency = sum(t.count(symbol_str) for t in targets)
            if frequency:
                counts.append((frequency, symbol))
        counts.sort(key=lambda pair: pair[0], reverse=True)
        return [value for _, value in counts[:limit]]

    def _candidate_deflector(self, node: int, cache: dict[str, Any], affected: Sequence[int]) -> list[tuple[int, Any]]:
        table = self.brain.deflectors_list
        cur_dest, cur_value = table[node]
        probe_rows = self._sample_rows(affected, min(self.proposal_rows, 32))
        target_symbols = self._target_symbols(probe_rows, cache, limit=4)

        integer_values = [v for v, _ in self.integer_counts.most_common(2)]
        values: list[Any] = [cur_value] + integer_values + target_symbols
        if self.brain.output and self.rng.random() < 0.8:
            choices = [x for x in self.brain.output if not isinstance(x, int)]
            if choices:
                values.append(self.rng.choice(choices))
        dedup_values = []
        seen_values = set()
        for value in values:
            key = self._value_key(value)
            if key not in seen_values:
                seen_values.add(key)
                dedup_values.append(value)
        values = dedup_values[:6]

        destinations = [cur_dest]
        for delta in (-8, -4, -2, -1, 1, 2, 4, 8):
            candidate = node + delta
            if 0 <= candidate < self.D:
                destinations.append(candidate)
        # Frequently touched nodes are useful destination proposals because they
        # already lie on trajectories; hole nodes matching target symbols are useful
        # because a single transition can expose a desired output symbol.
        active_nodes = cache["active_nodes"]
        sampled_active = self.rng.sample(active_nodes, min(16, len(active_nodes))) if active_nodes else []
        traffic = sorted(sampled_active, key=lambda d: len(cache["node_rows"][d]), reverse=True)
        destinations.extend(traffic[:4])
        matching_holes = []
        for symbol in target_symbols:
            matching_holes.extend(islice(iter(self.hole_nodes.get(self._value_key(symbol), set())), 4))
        self.rng.shuffle(matching_holes)
        destinations.extend(matching_holes[:4])
        destinations.extend(self.rng.randrange(self.D) for _ in range(3))
        dedup_destinations = []
        seen_dest = set()
        for dest in destinations:
            if 0 <= dest < self.D and dest not in seen_dest:
                seen_dest.add(dest)
                dedup_destinations.append(dest)
        destinations = dedup_destinations

        candidates: list[tuple[int, Any]] = []
        seen = set()
        def add(dest: int, val: Any) -> None:
            key = (int(dest), self._value_key(val))
            current = (int(cur_dest), self._value_key(cur_value))
            if key != current and key not in seen:
                seen.add(key)
                candidates.append((int(dest), val))
        # Make sure both value-only and destination-only alternatives are considered.
        for val in values:
            add(cur_dest, val)
        for dest in destinations:
            add(dest, cur_value)
        for dest in destinations[:5]:
            for val in values[:4]:
                add(dest, val)
        self.rng.shuffle(candidates)
        return candidates[: self.candidate_budget * 3]

    def _candidate_start_position(
        self, alpha_id: int, cache: dict[str, Any], affected: Sequence[int]
    ) -> list[int]:
        current = int(self.brain.alphabet_start_positions[alpha_id])
        candidate_positions = [current]
        probe_rows = self._sample_rows(affected, min(self.proposal_rows, 24))
        target_symbols = self._target_symbols(probe_rows, cache, limit=5)
        interesting_nodes = []
        for symbol in target_symbols:
            interesting_nodes.extend(islice(iter(self.hole_nodes.get(self._value_key(symbol), set())), 4))
        self.rng.shuffle(interesting_nodes)
        interesting_nodes = interesting_nodes[:8]
        active_nodes = cache["active_nodes"]
        sampled_active = self.rng.sample(active_nodes, min(16, len(active_nodes))) if active_nodes else []
        traffic = sorted(sampled_active, key=lambda d: len(cache["node_rows"][d]), reverse=True)
        interesting_nodes.extend(traffic[:6])
        # If an occurrence of this token starts at s + offset, set s so that it
        # arrives at a target-relevant node. This proposal is exact algebra, not a
        # relaxation of the discrete simulation.
        for local_row in probe_rows[:12]:
            example = self.cases[cache["case_ids"][local_row]]
            token_ids = example["token_ids"]
            offsets = self._context_offsets(token_ids)
            for i, aid in enumerate(token_ids):
                if aid != alpha_id:
                    continue
                for node in interesting_nodes[:8]:
                    candidate_positions.append((node - offsets[i]) % self.D)
        candidate_positions.extend(self.rng.randrange(self.D) for _ in range(8))
        unique = list(dict.fromkeys(int(p) % self.D for p in candidate_positions))
        self.rng.shuffle(unique)
        if current in unique:
            unique.remove(current)
            unique.insert(0, current)
        return unique[: self.candidate_budget * 2]

    def _candidate_direction(self, alpha_id: int) -> list[int]:
        current = int(self.brain.alphabet_directions[alpha_id])
        bound = int(self.brain.direction_range)
        if 2 * bound + 1 <= 64:
            values = list(range(-bound, bound + 1))
        else:
            values = [current, 0, -bound, bound]
            for delta in (-16, -8, -4, -2, -1, 1, 2, 4, 8, 16):
                values.append(max(-bound, min(bound, current + delta)))
            values.extend(self.rng.randint(-bound, bound) for _ in range(16))
        return list(dict.fromkeys(v for v in values if v != current))[: self.candidate_budget * 2]

    def _set_parameter(self, kind: str, index: int, value: Any) -> None:
        if kind == "deflector":
            dest, symbol = value
            self.brain.deflectors_list[index][0] = int(dest)
            self.brain.deflectors_list[index][1] = symbol
        elif kind == "start":
            self.brain.alphabet_start_positions[index] = int(value)
        elif kind == "direction":
            self.brain.alphabet_directions[index] = int(value)
        else:
            raise ValueError(kind)

    def _current_parameter(self, kind: str, index: int) -> Any:
        if kind == "deflector":
            d, v = self.brain.deflectors_list[index]
            return (d, v)
        if kind == "start":
            return self.brain.alphabet_start_positions[index]
        return self.brain.alphabet_directions[index]

    def _one_move(self, cache: dict[str, Any], temperature: float) -> bool:
        # 80% of proposals target graph nodes; 20% target token encoding parameters.
        choice = self.rng.random()
        active_nodes = cache["active_nodes"]
        alpha_counts = Counter()
        for local_row, case_id in enumerate(cache["case_ids"]):
            for alpha_id in self.cases[case_id]["alpha_set"]:
                alpha_counts[alpha_id] += 1
        active_alphas = [a for a, count in alpha_counts.items() if count]

        if choice < 0.80 and active_nodes:
            # Bias toward parameters that influence more rows, but retain exploration
            # of less common active nodes.
            probe_nodes = self.rng.sample(active_nodes, min(4, len(active_nodes)))
            if self.rng.random() < 0.75:
                weights = [math.sqrt(len(cache["node_rows"][d])) for d in probe_nodes]
                node = self.rng.choices(probe_nodes, weights=weights, k=1)[0]
            else:
                node = self.rng.choice(probe_nodes)
            affected = sorted(cache["node_rows"][node])
            candidates = self._candidate_deflector(node, cache, affected)
            kind, index = "deflector", node
        elif active_alphas:
            alpha_id = self.rng.choice(active_alphas)
            affected = [r for r, case_id in enumerate(cache["case_ids"])
                        if alpha_id in self.cases[case_id]["alpha_set"]]
            if choice < 0.90:
                candidates = self._candidate_start_position(alpha_id, cache, affected)
                kind, index = "start", alpha_id
            else:
                candidates = self._candidate_direction(alpha_id)
                kind, index = "direction", alpha_id
        else:
            return False

        if not affected or not candidates:
            return False
        self.steps_taken += 1
        current_value = self._current_parameter(kind, index)
        probe_rows = self._sample_rows(affected, self.proposal_rows)
        if not probe_rows:
            return False
        # Deduplicate candidates while preserving their values/types.
        deduped = []
        seen = set()
        for candidate in candidates:
            key = (candidate[0], self._value_key(candidate[1])) if kind == "deflector" else self._value_key(candidate)
            current_key = ((current_value[0], self._value_key(current_value[1]))
                           if kind == "deflector" else self._value_key(current_value))
            if key == current_key or key in seen:
                continue
            seen.add(key)
            deduped.append(candidate)
        candidates = deduped
        if len(candidates) > self.candidate_budget:
            # Candidate generators put some useful alternatives first, but randomize
            # their tail so long runs do not repeatedly inspect the same tiny subset.
            head_count = max(2, self.candidate_budget // 3)
            head = candidates[:head_count]
            tail = candidates[head_count:]
            candidates = head + self.rng.sample(tail, min(len(tail), self.candidate_budget - len(head)))

        quick_scores = []
        for candidate in candidates:
            self._set_parameter(kind, index, candidate)
            score = 0.0
            for row in probe_rows:
                case = self.cases[cache["case_ids"][row]]
                prediction, _ = self._simulate_prepared(case, collect_nodes=False)
                score += normalized_edit_distance(prediction, case["target"])
            score /= len(probe_rows)
            quick_scores.append((score, candidate))
        self._set_parameter(kind, index, current_value)
        quick_scores.sort(key=lambda item: item[0])
        finalists = [candidate for _, candidate in quick_scores[: self.finalists]]
        if not finalists:
            return False

        baseline_full = sum(cache["losses"][r] for r in affected) / len(affected)
        full_scores = []
        for candidate in finalists:
            self._set_parameter(kind, index, candidate)
            total = 0.0
            for row in affected:
                case = self.cases[cache["case_ids"][row]]
                prediction, _ = self._simulate_prepared(case, collect_nodes=False)
                total += normalized_edit_distance(prediction, case["target"])
            full_scores.append((total / len(affected), candidate))
        self._set_parameter(kind, index, current_value)
        full_scores.sort(key=lambda item: item[0])
        new_loss, new_value = full_scores[0]
        delta = new_loss - baseline_full
        accept = delta < -1e-12
        if not accept and abs(delta) <= 1e-12:
            accept = self.rng.random() < self.neutral_move_probability
        elif not accept and temperature > 0 and delta > 0:
            accept = self.rng.random() < math.exp(-delta / max(temperature, 1e-12))
        if accept:
            if kind == "deflector":
                old_symbol = self.brain.deflectors_list[index][1]
                new_symbol = new_value[1]
                if self._value_key(old_symbol) != self._value_key(new_symbol):
                    if isinstance(old_symbol, int):
                        self.integer_counts[old_symbol] -= 1
                        if self.integer_counts[old_symbol] <= 0:
                            del self.integer_counts[old_symbol]
                    if isinstance(new_symbol, int):
                        self.integer_counts[new_symbol] += 1
                    else:
                        self.hole_nodes.setdefault(self._value_key(new_symbol), set()).add(index)
                    if not isinstance(old_symbol, int):
                        old_nodes = self.hole_nodes.get(self._value_key(old_symbol))
                        if old_nodes is not None:
                            old_nodes.discard(index)
                            if not old_nodes:
                                self.hole_nodes.pop(self._value_key(old_symbol), None)
            self._set_parameter(kind, index, new_value)
            self._refresh_rows(cache, affected)
            self.accepted_moves += 1
            return True
        return False

    def _evaluate_prepared(self, prepared: Sequence[dict[str, Any]]) -> dict[str, Any]:
        if not prepared:
            return {"n": 0, "loss": float("nan"), "exact_match": float("nan")}
        total_loss = 0.0
        exact = 0
        for example in prepared:
            prediction, _ = self._simulate_prepared(example)
            target = example["target"]
            total_loss += normalized_edit_distance(prediction, target)
            exact += prediction == target
        return {"n": len(prepared), "loss": total_loss / len(prepared),
                "exact_match": exact / len(prepared)}

    def fit(
        self,
        *,
        epochs: int = 10,
        validation_cases: Optional[Iterable[tuple[Any, Any]]] = None,
        batches_per_epoch: int = 24,
        steps_per_batch: int = 8,
        verbose: bool = True,
        restore_best: bool = True,
    ) -> dict[str, Any]:
        """Train in place and optionally restore the best held-out checkpoint.

        The optimizer holds trajectories only for one minibatch at a time. With a
        fixed batch_size, optimization working memory is O(batch_size * trace size
        + number_of_deflectors), not O(dataset size * trace size).
        """
        epochs = max(1, int(epochs))
        batches_per_epoch = max(1, int(batches_per_epoch))
        steps_per_batch = max(1, int(steps_per_batch))
        validation = None if validation_cases is None else self._prepare_cases(validation_cases)
        monitor = validation if validation is not None else self.cases
        best_snapshot = _brain_snapshot(self.brain)
        best_metrics = self._evaluate_prepared(monitor)
        best_key = (best_metrics["loss"], -best_metrics["exact_match"])
        total_planned_steps = epochs * batches_per_epoch * steps_per_batch
        step_at_start = self.steps_taken

        for epoch in range(epochs):
            epoch_start_accepted = self.accepted_moves
            for _ in range(batches_per_epoch):
                n = min(self.batch_size, len(self.cases))
                case_ids = self.rng.sample(range(len(self.cases)), n)
                cache = self._make_batch_cache(case_ids)
                for _ in range(steps_per_batch):
                    elapsed = self.steps_taken - step_at_start
                    fraction = min(1.0, elapsed / max(1, total_planned_steps))
                    temperature = self.initial_temperature * (1.0 - fraction)
                    self._one_move(cache, temperature)

            metrics = self._evaluate_prepared(monitor)
            key = (metrics["loss"], -metrics["exact_match"])
            record = {
                "epoch": epoch + 1,
                "monitor": "validation" if validation is not None else "training",
                "loss": metrics["loss"],
                "exact_match": metrics["exact_match"],
                "accepted_moves_this_epoch": self.accepted_moves - epoch_start_accepted,
                "accepted_moves_total": self.accepted_moves,
                "steps_total": self.steps_taken,
            }
            self.history.append(record)
            if key < best_key:
                best_key = key
                best_metrics = metrics
                best_snapshot = _brain_snapshot(self.brain)
            if verbose:
                print(
                    f"epoch {epoch + 1:>3}/{epochs}: "
                    f"{record['monitor']} loss={metrics['loss']:.4f}, "
                    f"exact={metrics['exact_match']:.3%}, "
                    f"accepted={record['accepted_moves_this_epoch']}"
                )
        if restore_best:
            _restore_snapshot(self.brain, best_snapshot)
            self._rebuild_value_indexes()
        return {
            "best_metrics": best_metrics,
            "history": list(self.history),
            "accepted_moves": self.accepted_moves,
            "steps": self.steps_taken,
            "monitor": "validation" if validation is not None else "training",
        }


def search_brain_sizes(
    brain_factory: Callable[[int], Any],
    sizes: Sequence[int],
    train_cases: Iterable[tuple[Any, Any]],
    validation_cases: Iterable[tuple[Any, Any]],
    *,
    tolerance: float = 0.01,
    seed: int = 0,
    restarts_per_size: int = 1,
    trainer_kwargs: Optional[dict[str, Any]] = None,
    fit_kwargs: Optional[dict[str, Any]] = None,
    verbose: bool = True,
) -> tuple[Any, list[dict[str, Any]]]:
    """Train a size sweep and return the smallest tested size near best val loss.

    `brain_factory(D)` must return a fresh instance of the unchanged architecture
    with D deflectors. Since this architecture is not provably nested as D varies,
    this evaluates every supplied size rather than binary-searching under a false
    monotonicity assumption. Optional restarts reduce sensitivity to random
    initialization. The result is the smallest qualifying size *among tested sizes
    and restarts*, not a proof of the global mathematical optimum.
    """
    train_cases = list(train_cases)
    validation_cases = list(validation_cases)
    if not sizes:
        raise ValueError("Provide at least one candidate size.")
    restarts_per_size = max(1, int(restarts_per_size))
    trainer_kwargs = dict(trainer_kwargs or {})
    fit_kwargs = dict(fit_kwargs or {})
    runs = []
    snapshots = []
    for size in sorted(set(int(s) for s in sizes)):
        if size < 1:
            continue
        restart_results = []
        best_brain = None
        best_snapshot = None
        best_key = (float("inf"), float("inf"))
        best_record = None
        for restart in range(restarts_per_size):
            run_seed = seed + size * 1009 + restart * 9176
            # Keep model initialization reproducible without disturbing caller RNG.
            global_state = random.getstate()
            random.seed(run_seed)
            try:
                brain = brain_factory(size)
            finally:
                random.setstate(global_state)
            run_trainer_kwargs = dict(trainer_kwargs)
            trainer_seed = run_trainer_kwargs.pop("seed", seed + size + restart)
            trainer = SparseTraceTrainer(
                brain, train_cases, seed=trainer_seed + restart, **run_trainer_kwargs
            )
            run_fit_kwargs = dict(fit_kwargs)
            run_fit_kwargs["validation_cases"] = validation_cases
            run_fit_kwargs["verbose"] = verbose
            train_result = trainer.fit(**run_fit_kwargs)
            val_metrics = trainer.evaluate(validation_cases)
            key = (val_metrics["loss"], -val_metrics["exact_match"])
            restart_record = {
                "restart": restart,
                **val_metrics,
                "train_best": train_result["best_metrics"],
                "accepted_moves": train_result["accepted_moves"],
            }
            restart_results.append(restart_record)
            if key < best_key:
                best_key = key
                best_brain = brain
                best_snapshot = _brain_snapshot(brain)
                best_record = restart_record
        assert best_brain is not None and best_snapshot is not None and best_record is not None
        _restore_snapshot(best_brain, best_snapshot)
        record = {
            "size": size,
            "loss": best_record["loss"],
            "exact_match": best_record["exact_match"],
            "best_restart": best_record["restart"],
            "train_best": best_record["train_best"],
            "accepted_moves": best_record["accepted_moves"],
            "restarts": restart_results,
        }
        runs.append(record)
        snapshots.append((best_snapshot, best_brain))
        if verbose:
            print(
                f"size={size}: validation loss={record['loss']:.4f}, "
                f"exact={record['exact_match']:.3%}, "
                f"best restart={record['best_restart']}"
            )

    if not runs:
        raise ValueError("No valid candidate sizes were supplied.")
    best_loss = min(row["loss"] for row in runs)
    threshold = best_loss + max(0.0, float(tolerance))
    qualifying = [row for row in runs if row["loss"] <= threshold]
    chosen_size = min(qualifying, key=lambda row: row["size"])["size"]
    chosen_index = next(i for i, row in enumerate(runs) if row["size"] == chosen_size)
    chosen_brain = snapshots[chosen_index][1]
    _restore_snapshot(chosen_brain, snapshots[chosen_index][0])
    return chosen_brain, runs


## 1. Smoke test: verify exact simulator parity

This small example is only a sanity check that the trainer reproduces the original simulator and runs end-to-end. Its random targets are not intended to be an easy or meaningful benchmark.


In [ ]:
import random

random.seed(7)
alphabet = list("abc")
output = list("xyz")
brain = Ball_Trap_Brain(40, 3, alphabet, output, holes_multiplier=1, direction_range=5)

smoke_cases = [
    ("a", "x"), ("b", "y"), ("c", "z"),
    ("ab", "xy"), ("bc", "yz"), ("ca", "zx"),
    ("abc", "xyz"), ("cba", "zyx"), ("ac", "xz"), ("ba", "yx"),
]

trainer = SparseTraceTrainer(
    brain, smoke_cases, batch_size=8, candidate_budget=8,
    proposal_rows=5, finalists=1, seed=7,
)
assert trainer.verify_simulator()
print("Simulator parity: PASS")
print("Before:", trainer.evaluate())

smoke_result = trainer.fit(
    epochs=2, validation_cases=smoke_cases[-2:],
    batches_per_epoch=3, steps_per_batch=4, verbose=True,
)
assert trainer.verify_simulator()
print("After:", trainer.evaluate())
print("Validation checkpoint:", smoke_result["best_metrics"])


## 2. Train your dataset

Replace the example pairs below with your own `(input, expected_output)` pairs. Split by **examples or underlying source records**, not by duplicated copies of the same input. Keep a validation set separate from the cases passed to the trainer. Do not tune model size against your final test set.

You can start with conservative settings. Increase the number of discrete proposals before increasing batch size if compute is tight.


In [ ]:
# Replace these examples with your actual data. Targets must be strings, as in the original simulator.
train_cases = [("a", "x"), ("b", "y"), ("ab", "xy"), ("ba", "yx")]
validation_cases = [("c", "z"), ("abc", "xyz")]

# Create a fresh brain with the exact original architecture.
brain = Ball_Trap_Brain(128, 4, list("abc"), list("xyz"), holes_multiplier=1, direction_range=10)
trainer = SparseTraceTrainer(
    brain, train_cases,
    batch_size=64,          # working-set bound
    candidate_budget=24,    # alternatives screened per parameter update
    proposal_rows=24,       # examples used for shortlisting
    finalists=2,            # finalists checked on every affected row in the batch
    temperature=0.025,      # small probability of accepting a temporary worsening
    seed=42,
)

trainer.verify_simulator()  # should not raise
result = trainer.fit(
    epochs=10,
    validation_cases=validation_cases,
    batches_per_epoch=24,
    steps_per_batch=8,
    verbose=True,
)
print("Held-out:", trainer.evaluate(validation_cases))
print("Best monitor metrics:", result["best_metrics"])


## 3. Search for an appropriately small brain

Because changing the deflector count changes modulo arithmetic and geometric travel distances, a brain with `D+1` deflectors is not necessarily a strict superset of a brain with `D` deflectors. Give the search a range of candidate sizes that you can afford to evaluate. It returns the **smallest tested size** whose validation loss is within `tolerance` of the best tested size.

The code below is an example to adapt; run it after replacing `train_cases` and `validation_cases` above.


In [ ]:
# Example (uncomment/adapt):
# alphabet = list("abc")
# output = list("xyz")
# def make_brain(D):
#     return Ball_Trap_Brain(D, 4, alphabet, output, holes_multiplier=1, direction_range=10)
#
# best_brain, size_results = search_brain_sizes(
#     make_brain, sizes=[32, 64, 128, 256],
#     train_cases=train_cases, validation_cases=validation_cases,
#     tolerance=0.01, seed=42,
#     trainer_kwargs={
#         "batch_size": 64, "candidate_budget": 24,
#         "proposal_rows": 24, "finalists": 2,
#     },
#     fit_kwargs={
#         "epochs": 10, "batches_per_epoch": 24, "steps_per_batch": 8,
#     },
# )
# print(size_results)
# print("Selected deflectors:", len(best_brain.deflectors_list))


## Practical tuning notes

- **Too slow:** reduce `candidate_budget`, `proposal_rows`, `finalists`, or `steps_per_batch`. Those directly bound candidate-evaluation work.
- **Stagnating:** increase `candidate_budget`, use multiple random seeds, raise `batches_per_epoch`, or use a broader set of candidate sizes. This remains heuristic search and can get stuck in local optima.
- **Training score improves but validation does not:** use more examples, avoid duplicate leakage across the split, and choose size by validation rather than training loss.
- **Exact match matters most:** monitor `exact_match` as well as normalized edit loss. The normalized edit distance supplies a useful graded objective when an output is close but not exact.
- **Memory:** cached trajectory traces are limited to one batch; model parameters themselves still require O(D) storage. Token IDs are packed into 16- or 32-bit arrays, but the prepared dataset inputs and targets still have to be available to the trainer.

The current implementation retains the original simulator's fixed ricochet cap of 10 and uses its existing discrete integer directions, starting positions, destinations, values, and energy rules.
